# 03. DEM phase deramping

This notebook isolates the single deramp operation inside interferogram formation: multiplying the secondary SLC by `exp(1j * dem_phase)` to remove the known reference (flat-earth plus topographic) phase. It shows that the residual interferometric phase after deramping reflects only the height deviation from the DEM, not the absolute topography.

**Modules exercised:** pipelines.processing_pipeline.interferogram (deramp step in _compute_interferograms)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Setup: a sloped scene with a bump

We model a scene whose true height is a smooth ramp (the part captured by the DEM) plus a localised bump (the part the DEM does not know about). The interferometric phase before deramping mixes both; after deramping only the bump should remain.

In [ ]:
n_az, n_rg = 48, 80
kz_track   = 0.20

az = np.arange(n_az)[:, None]
rg = np.arange(n_rg)[None, :]

dem_height  = 0.4 * rg + 0.0 * az
bump        = 12.0 * np.exp(-(((az - 24) ** 2 + (rg - 40) ** 2) / (2.0 * 5.0 ** 2)))
true_height = dem_height + bump
print('dem range:', float(dem_height.min()), float(dem_height.max()))
print('bump peak:', float(bump.max()))

## Form the secondary SLC and its DEM phase

The secondary carries phase `kz * true_height`. The DEM phase the pipeline removes is `kz * dem_height`, i.e. the known part only.

In [ ]:
amplitude  = np.ones((n_az, n_rg), dtype=np.float32)
secondary  = (amplitude * np.exp(1.0j * kz_track * true_height)).astype(np.complex64)
primary    = (amplitude * np.exp(1.0j * 0.0)).astype(np.complex64)
dem_phase  = kz_track * dem_height

phase_before = np.angle(primary * np.conj(secondary))
deramped     = secondary * np.exp(1.0j * dem_phase)
phase_after  = np.angle(primary * np.conj(deramped))
print('phase_before range:', float(phase_before.min()), float(phase_before.max()))
print('phase_after  range:', float(phase_after.min()),  float(phase_after.max()))

## Visual confirmation

Before deramping the phase should show ramp fringes across range plus the bump. After deramping the ramp fringes should be gone, leaving the localised bump signature.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
im0 = axes[0].imshow(true_height, aspect='auto', origin='lower')
axes[0].set_title('true height (DEM ramp + bump)')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)
im1 = axes[1].imshow(phase_before, cmap='twilight', vmin=-np.pi, vmax=np.pi, aspect='auto', origin='lower')
axes[1].set_title('phase before deramp')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)
im2 = axes[2].imshow(phase_after, cmap='twilight', vmin=-np.pi, vmax=np.pi, aspect='auto', origin='lower')
axes[2].set_title('phase after deramp')
for ax in axes:
    ax.set_xlabel('range'); ax.set_ylabel('azimuth')
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)
fig.tight_layout()
plt.show()

## Residual phase along a range cut

A 1D cut through the bump centre makes the effect explicit: the deramped residual should be near zero away from the bump and peak at the bump location.

In [ ]:
row = 24
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(phase_before[row], label='before deramp')
ax.plot(phase_after[row],  label='after deramp')
ax.axhline(0.0, color='0.6', lw=0.8)
ax.set_xlabel('range'); ax.set_ylabel('phase (rad)')
ax.set_title(f'range cut at azimuth {row}')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

The before-deramp phase wraps repeatedly across range due to the DEM ramp. The after-deramp phase is flat away from the bump and shows a clean localised deviation at the bump, confirming the deramp removes the known topographic phase and leaves only the unmodelled height residual.